# Laboratorio 9 - Inteligencia Artificial 

#### Sebastian Juárez - 21471
Link al repo: https://github.com/SebasJuarez/CC3045/tree/Lab9

### Task 1 - Teoría

**1. Diga cual es la diferencia entre Modelos de Markov y Hidden Markov Models**

Un Modelo de Markov es un sistema donde el siguiente estado se va calculando en base al estado actual, dejando a un lado el pasado. Ademas, los estados se pueden observar directamente, como por ejemplo, en un sistema de clima se puede ver como "soleado" o "lluvioso.

Un Hidden Markov Model (HMM) también sigue el metodo de que "solo importa el estado actual", pero los estados no se pueden observan directamente. Lo que vemos son ciertas señales u observaciones que nos dan pistas del estado oculto como por ejemplo, en este sistema de clima no se puede ver directamente si es soleado o lluvioso, solo se ve si es que la gente lleva paraguas o no y con eso se puede deducir el tiempo.

**2. Investigue qué son los factorial HMM (Hidden Markov Models)**

Un Factorial Hidden Markov Model (FHMM) es una extensión de un HMM.
En lugar de un solo estado oculto controlando las observaciones, hay varios estados HMM trabajando juntos, y todos contribuyen a generar la observación. Todos estos factores pueden estar pensados para observar diferentes caracteristicas haciendo que cualquier cambio en la cadena, hace que cambie el factor que se observa.

**3. Especifique en sus propias palabras el algoritmo Forward Backward para HMM**

El algoritmo Forward-Backward sirve para calcular la probabilidad de estar en un estado oculto en cierto momento tomando en cuenta todas las observaciones que se tienen a disposicion, como en el pasado y en el futuro. Estas se pueden explicar de mejor manera así:

En el Forward se calcula la probabilidad de llegar a cada estado en el "presente" usando todas las observaciones que hay desde el "pasado" hasta ahora.

en cambio, el Backward calcula la probabilidad de que a partir de un estado actual o en el presente se puedan explicar las observaciones en el futuro. Mas o menos como basado en el pasado, miramos al futuro

El resultado final combina las dos probabilidades para saber qué tan probable es que estés en un estado en un momento específico.

**4. En el algoritmo de Forward Backward, por qué es necesario el paso de Backward (puede escribir ejemplos o casos para responder esta pregunta)**

El paso Backward es necesario porque solo ver el pasado no es suficiente. Un estado podría parecer probable viendo el pasado, pero puede no ser real con lo que pasa después, como por ejemplo, si vemos a alguien con un paraguas (por que forward nos dijo que probablemente llueve) y luego miramos que hay sol y calor todo el dia, en este caso backward te corregiria diciendo que quizas la persona solo llevaba el paraguas por alguna otra razon, no por la lluvia.

Por esto es que el backward asegura que todo sea "coherente" con lo que se ve, sea antes o despues

### Task 2 - Código

Usando el codigo proporcionado en el laboratorio como referencia, definimos parametros, creamos clases y analizamos las probabilidades calculadas usando el algoritmo Forward Backward en HMM. En este caso definimos observaciones random para simular el clima real en que puede ir variando de manera aleatoria

In [ ]:
import random

class HMM:
    def __init__(self, states, observations, initial_prob, transition_prob, emission_prob):
        # Inicializar parámetros del HMM
        self.states = states
        self.observations = observations
        self.initial_prob = initial_prob
        self.transition_prob = transition_prob
        self.emission_prob = emission_prob

    def generate_sequence(self, length):
        # Generar una secuencia de observaciones basada en el HMM
        current_state = random.choices(self.states, weights=[self.initial_prob[s] for s in self.states])[0]
        sequence = []

        for _ in range(length):
            observation = random.choices(self.states, weights=[self.emission_prob[current_state][o] for o in self.states])[0]
            sequence.append(observation)
            current_state = random.choices(self.states, weights=[self.transition_prob[current_state][s] for s in self.states])[0]
        
        return sequence

    def forward(self, observations):
        # Paso hacia adelante
        fwd = [{}]

        for state in self.states:
            fwd[0][state] = self.initial_prob[state] * self.emission_prob[state][observations[0]]

        for t in range(1, len(observations)):
            fwd.append({})
            for curr_state in self.states:
                fwd[t][curr_state] = sum(
                    fwd[t-1][prev_state] * self.transition_prob[prev_state][curr_state]
                    for prev_state in self.states
                ) * self.emission_prob[curr_state][observations[t]]
        
        return fwd

    def backward(self, observations):
        # Paso hacia atrás
        bwd = [{} for _ in range(len(observations))]

        for state in self.states:
            bwd[-1][state] = 1 

        for t in reversed(range(len(observations) - 1)):
            for curr_state in self.states:
                bwd[t][curr_state] = sum(
                    self.transition_prob[curr_state][next_state] *
                    self.emission_prob[next_state][observations[t+1]] *
                    bwd[t+1][next_state]
                    for next_state in self.states
                )
        
        return bwd

    def compute_state_probabilities(self, observations):
        # Combinar forward y backward
        fwd = self.forward(observations)
        bwd = self.backward(observations)
        probs = []

        for t in range(len(observations)):
            total = sum(fwd[t][s] * bwd[t][s] for s in self.states)
            probs.append({s: (fwd[t][s] * bwd[t][s]) / total for s in self.states})
        
        return probs

# Definición de parámetros

states = ['Sunny', 'Rainy']
observations = ['Sunny', 'Sunny', 'Rainy']

initial_prob = {'Sunny': 0.5, 'Rainy': 0.5}

transition_prob = {
    'Sunny': {'Sunny': 0.8, 'Rainy': 0.2},
    'Rainy': {'Sunny': 0.4, 'Rainy': 0.6}
}

emission_prob = {
    'Sunny': {'Sunny': 0.9, 'Rainy': 0.1},
    'Rainy': {'Sunny': 0.2, 'Rainy': 0.7}
}

# Crear modelo HMM

hmm = HMM(states, observations, initial_prob, transition_prob, emission_prob)

# Generar secuencia de observaciones para probar
obs_sequence = hmm.generate_sequence(4)
print("Secuencia generada:", obs_sequence)

# Calcular probabilidades forward usando la secuencia generada aleatoriamente
forward_probs = hmm.forward(obs_sequence)
print("\nProbabilidades Forward:")
for t, probs in enumerate(forward_probs):
    print(f"Tiempo {t}:")
    for state in probs:
        prob = probs[state]
        print(f"  {state}: {prob:.6f} ({prob*100:.2f}%)")

# Calcular probabilidades backward usando la secuencia generada aleatoriamente
backward_probs = hmm.backward(obs_sequence)
print("\nProbabilidades Backward:")
for t, probs in enumerate(backward_probs):
    print(f"Tiempo {t}:")
    for state in probs:
        prob = probs[state]
        print(f"  {state}: {prob:.6f} ({prob*100:.2f}%)")

# Calcular probabilidades de estados combinando forward y backward
state_probs = hmm.compute_state_probabilities(obs_sequence)
print("\nProbabilidades de Estados (Combinando Forward y Backward):")
for t, probs in enumerate(state_probs):
    print(f"Tiempo {t}:")
    for state in probs:
        prob = probs[state]
        print(f"  {state}: {prob:.6f} ({prob*100:.2f}%)")



Secuencia generada: ['Rainy', 'Sunny', 'Rainy', 'Sunny']

Probabilidades Forward:
Tiempo 0:
  Sunny: 0.050000 (5.00%)
  Rainy: 0.350000 (35.00%)
Tiempo 1:
  Sunny: 0.162000 (16.20%)
  Rainy: 0.044000 (4.40%)
Tiempo 2:
  Sunny: 0.014720 (1.47%)
  Rainy: 0.041160 (4.12%)
Tiempo 3:
  Sunny: 0.025416 (2.54%)
  Rainy: 0.005528 (0.55%)

Probabilidades Backward:
Tiempo 0:
  Sunny: 0.101440 (10.14%)
  Rainy: 0.073920 (7.39%)
Tiempo 1:
  Sunny: 0.128000 (12.80%)
  Rainy: 0.232000 (23.20%)
Tiempo 2:
  Sunny: 0.760000 (76.00%)
  Rainy: 0.480000 (48.00%)
Tiempo 3:
  Sunny: 1.000000 (100.00%)
  Rainy: 1.000000 (100.00%)

Probabilidades de Estados (Combinando Forward y Backward):
Tiempo 0:
  Sunny: 0.163909 (16.39%)
  Rainy: 0.836091 (83.61%)
Tiempo 1:
  Sunny: 0.670114 (67.01%)
  Rainy: 0.329886 (32.99%)
Tiempo 2:
  Sunny: 0.361531 (36.15%)
  Rainy: 0.638469 (63.85%)
Tiempo 3:
  Sunny: 0.821355 (82.14%)
  Rainy: 0.178645 (17.86%)
